
# DESC SN Ia metric # 



In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

import healpy as hp
import pandas as pd

import rubin_sim.maf as maf
from rubin_sim.data import get_baseline
import time

## Configuration

In [ ]:
# os.environ["RUBIN_SIM_DATA_DIR"] = "/users/dagoret/DATA/OpSim"
path_topdir = os.getenv("RUBIN_SIM_DATA_DIR")
print(f"path_topdir = {path_topdir}")

In [ ]:
# Baseline Survey
baseline_file = get_baseline()
runName = os.path.split(baseline_file)[-1].replace(".db", "")

print(runName)

In [ ]:
data_dir = None

if data_dir is None:
    import tempfile
    import os

    data_dir_itself = tempfile.TemporaryDirectory(prefix="00_maf_SNIa_FastNSIDE4", dir=os.getcwd())
    data_dir = data_dir_itself.name

print(f"Using the {data_dir_itself.name} directory for output of this notebook")

In [ ]:
# Set up MAF output
outDir = data_dir
resultsDb = maf.db.ResultsDb(out_dir=outDir)

In [ ]:
plotDict = {"percentileClip": 95.0, "nTicks": 5}

sne_nside = 4
sn_summary = [maf.MedianMetric(), maf.MeanMetric(), maf.SumMetric(metric_name="Total detected")]
slicer = maf.HealpixSlicer(nside=sne_nside, use_cache=False)
# slicer = maf.HealpixSubsetSlicer(nside=16, hpid=[890], useCache=False)
# slicer = maf.HealpixSubsetSlicer(nside=16, hpid=[889], useCache=False)

metric = maf.SNNSNMetric(verbose=False)
bundle = maf.MetricBundle(
    metric, slicer, None, plot_dict=plotDict, summary_metrics=sn_summary, run_name=runName
)

bg = maf.MetricBundleGroup({"sn": bundle}, baseline_file, outDir, resultsDb)

In [ ]:
t1 = time.time()
bg.run_all()
t2 = time.time()
bg.plot_all(closefigs=False)
print("runtime=", t2 - t1, "s")

In [ ]:
bundle.metric_values.compressed()

In [ ]:
# The 'reduce' values of the metric got stored in the bundle dict in the bungle group
bg.bundle_dict

In [ ]:
# The nSN and zlim values are pulled out in those reduce methods, into their own bundles.
bdict = bg.bundle_dict
print(bdict["SNNSNMetric_reducen_sn"].metric_values.compressed())
np.median(bdict["SNNSNMetric_reducen_sn"].metric_values.compressed())

In [ ]:
bdict["SNNSNMetric_reducezlim"].metric_values.compressed()